In [2]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from glob import glob
import os
import numpy as np
import tensorflow
#import torch
import pandas as pd
#import torchaudio

from tqdm.notebook import tqdm

print(tensorflow.config.list_physical_devices('GPU'))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
#! pip install git+https://github.com/openai/whisper.git
import whisper

In [4]:
mouseFLAC  = glob("vctk/gen/flac/wav48_silence_trimmed/*/*.flac")
mouseTXT = [m_fl.replace("_mic2.flac", ".txt").replace("vctk/gen/flac/wav48_silence_trimmed", "vctk/stock/txt") for m_fl in mouseFLAC]

mouseS = []
for i in mouseTXT:
    with open(i, 'r') as f:
        mouseS.append(f.read())

#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print("device: {}".format(device))
#torch.set_default_device(device)

In [5]:
import datasets
from datasets import Dataset, Audio

df = pd.DataFrame({"audio": mouseFLAC, "sentence": mouseS})
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))


/home/tsu/.miniforge3/envs/micemouse/lib/python3.9/site-packages/pyarrow/pandas_compat.py:373: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if _pandas_api.is_sparse(col):


In [6]:
#from huggingface_hub import notebook_login

#notebook_login()

In [7]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small", language="English", task="transcribe")



from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="English", task="transcribe")



In [8]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

do_lower_case = False
do_remove_punctuation = False

normalizer = BasicTextNormalizer()



#print(dataset["audio"][0][0])


#model = whisper.load_model("medium.en")




In [9]:
from transformers import WhisperProcessor

def prepare_dataset(batch):
    # load and (possibly) resample audio data to 16kHz
    audio = batch["audio"]

    # compute log-Mel input features from input audio array 
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    # compute input length of audio sample in seconds
    batch["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    
    # optional pre-processing steps
    transcription = batch["sentence"]
    if do_lower_case:
        transcription = transcription.lower()
    if do_remove_punctuation:
        transcription = normalizer(transcription).strip()
    
    # encode target text to label ids
    batch["labels"] = processor.tokenizer(transcription).input_ids
    return batch



In [10]:


dataset = dataset.map(prepare_dataset, num_proc=2)



Map (num_proc=2):   0%|          | 0/3835 [00:00<?, ? examples/s]

In [18]:


import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch



In [19]:


import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch



In [20]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)


In [21]:
import evaluate

metric = evaluate.load("wer")


In [22]:
# evaluate with the 'normalised' WER
do_normalize_eval = True

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    if do_normalize_eval:
        pred_str = [normalizer(pred) for pred in pred_str]
        label_str = [normalizer(label) for label in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [23]:


from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")



In [24]:


model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False



In [32]:
from transformers import Seq2SeqTrainingArguments
import accelerate

training_args = Seq2SeqTrainingArguments(
    output_dir="./stt",
    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,  # increase by 2x for every 2x decrease in batch size
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=5000,
    gradient_checkpointing=True,
    fp16=True, # should be True, but having CUDA issues
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=1000,
    eval_steps=1000,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,
)

In [33]:


from transformers import Seq2SeqTrainer

dataset=dataset.train_test_split(test_size=0.1)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)



AttributeError: 'DatasetDict' object has no attribute 'train_test_split'

In [34]:


processor.save_pretrained(training_args.output_dir)



In [35]:
trainer.train()


AttributeError: 'AcceleratorState' object has no attribute 'distributed_type'

In [ ]:

# OLD CODE | DONT RUN

# n = 5

# for i in range(n):
#     PATH = mouseFLAC[i]
#     with open(mouseTXT[i], 'r') as f:
#         transcript = f.read().strip()
        
#         result = model.transcribe(PATH)
#         print("{} \t\t: {}".format( i, PATH))
#         print("real \t\t: {}".format(transcript))
#         print("transcription \t: {}".format(result["text"]))
#         print()